In [ ]:
import subprocess
import shutil
import psutil

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv",
    ],
    capture_output=True, text=True, check=True,
)
print(result.stdout)
print(f"System RAM: {psutil.virtual_memory().total / 2**30:.1f} GiB")
print(f"Free disk: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")

In [ ]:
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
assert (PROJECT / "notebooks/project_utils.py").is_file()

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "transformers", "accelerate", "safetensors", "huggingface_hub",
    ],
    check=True,
)

import torch
import transformers
from transformers import AutoConfig, AutoProcessor, AutoModelForMultimodalLM

sys.path.insert(0, str(PROJECT / "notebooks"))
from project_utils import new_run, record_environment

assert torch.cuda.is_available(), "Connect a GPU runtime."
assert torch.cuda.is_bf16_supported(), "BF16 support check failed."

# Verify that GPU computation works.
with torch.inference_mode():
    probe = torch.randn(128, 128, device="cuda", dtype=torch.bfloat16)
    probe_result = probe @ probe
    assert torch.isfinite(probe_result).all().item()

del probe, probe_result
torch.cuda.empty_cache()

GEMMA_ID = "google/gemma-4-31B-it"

# Reuse the exact checkpoint verified in our earlier setup.
GEMMA_REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"

gemma_config = AutoConfig.from_pretrained(
    GEMMA_ID,
    revision=GEMMA_REVISION,
)
assert gemma_config.model_type == "gemma4", gemma_config.model_type

GEMMA_RUN = new_run(PROJECT, "step04_gemma4_31b")
record_environment(
    PROJECT,
    GEMMA_RUN / "setup_manifest.json",
    model_id=GEMMA_ID,
    model_revision=GEMMA_REVISION,
    gpu=torch.cuda.get_device_name(0),
    planned_dtype="bfloat16",
    planned_thinking=False,
)

free_bytes, total_bytes = torch.cuda.mem_get_info()

print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {free_bytes / 2**30:.1f} GiB free / "
      f"{total_bytes / 2**30:.1f} GiB total")
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("BF16 GPU operation: PASS")
print("Model configuration:", gemma_config.model_type)
print("Pinned revision:", GEMMA_REVISION)
print("Run directory:", GEMMA_RUN)

In [ ]:
import time
import torch
from pathlib import Path
from transformers import AutoProcessor, AutoModelForMultimodalLM
from project_utils import save_json, record_environment

GEMMA_CACHE = Path("/content/hf_cache")
GEMMA_CACHE.mkdir(parents=True, exist_ok=True)

print("Loading checkpoint:", GEMMA_ID)
print("Revision:", GEMMA_REVISION)
print("Temporary model cache:", GEMMA_CACHE)

started = time.perf_counter()
torch.cuda.reset_peak_memory_stats()

gemma_processor = AutoProcessor.from_pretrained(
    GEMMA_ID,
    revision=GEMMA_REVISION,
    cache_dir=str(GEMMA_CACHE),
)

gemma_model = AutoModelForMultimodalLM.from_pretrained(
    GEMMA_ID,
    revision=GEMMA_REVISION,
    cache_dir=str(GEMMA_CACHE),
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
    use_safetensors=True,
).eval()

gemma_model.requires_grad_(False)
torch.cuda.synchronize()

assert all(
    parameter.device.type == "cuda"
    for parameter in gemma_model.parameters()
), "Some model parameters are not on the GPU."

free_bytes, total_bytes = torch.cuda.mem_get_info()
load_report = {
    "model_id": GEMMA_ID,
    "model_revision": GEMMA_REVISION,
    "model_class": type(gemma_model).__name__,
    "dtype": "bfloat16",
    "attention": "sdpa",
    "gpu": torch.cuda.get_device_name(0),
    "download_and_load_seconds": time.perf_counter() - started,
    "allocated_gpu_gib": torch.cuda.memory_allocated() / 2**30,
    "peak_allocated_gpu_gib": torch.cuda.max_memory_allocated() / 2**30,
    "free_gpu_gib": free_bytes / 2**30,
    "cache_directory": str(GEMMA_CACHE),
}

save_json(GEMMA_RUN / "model_load.json", load_report)
record_environment(
    PROJECT,
    GEMMA_RUN / "loaded_environment.json",
    model_id=GEMMA_ID,
    model_revision=GEMMA_REVISION,
)

print("\nMODEL LOADED")
print("Class:", load_report["model_class"])
print(f"GPU allocated: {load_report['allocated_gpu_gib']:.1f} GiB")
print(f"GPU free: {load_report['free_gpu_gib']:.1f} GiB")
print(f"Download + load: {load_report['download_and_load_seconds']:.1f}s")
print("Report:", GEMMA_RUN / "model_load.json")

In [ ]:
import sys
import torch

sys.path.insert(0, str(PROJECT / "src"))
from semeval_task3.data import load_dataset
from project_utils import save_json

# Load inputs without labels for this compatibility check.
gemma_dev = load_dataset(PROJECT / "work/data/dev", with_gold=False)
assert not any(u.span_flags for u in gemma_dev.usages)

GEMMA_SYSTEM_PROMPT = "You are an expert lexicographer of historical Swedish."

def gemma_messages(u):
    message = (
        f"Target word (lemma): {u.word}\n"
        f"Sense definition (Swedish): {u.definition}\n"
        f"Sentence from {u.year}: {u.marked_text()}\n\n"
        "Does the word marked with <t> and </t> in the sentence express "
        "the sense given by the definition? Answer only Yes or No."
    )
    return [
        {"role": "system", "content": GEMMA_SYSTEM_PROMPT},
        {"role": "user", "content": message},
    ]

gemma_tokenizer = gemma_processor.tokenizer
yes_ids = gemma_tokenizer.encode("Yes", add_special_tokens=False)
no_ids = gemma_tokenizer.encode("No", add_special_tokens=False)
print("Yes token IDs:", yes_ids)
print("No token IDs:", no_ids)
assert len(yes_ids) == len(no_ids) == 1, (
    "Answers are not single tokens. Stop and send this output."
)
GEMMA_YES_ID, GEMMA_NO_ID = yes_ids[0], no_ids[0]
assert GEMMA_YES_ID != GEMMA_NO_ID

# Two different words, selected without inspecting labels.
first = gemma_dev.usages[0]
second = next(u for u in gemma_dev.usages if u.group != first.group)
gemma_smoke_results = []

for u in [first, second]:
    messages = gemma_messages(u)
    inputs = gemma_processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to("cuda")

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = gemma_model(
            **inputs, use_cache=False, logits_to_keep=1
        )
        logits = output.logits[0, -1].float()
        assert torch.isfinite(logits).all().item()

        probabilities = torch.softmax(logits, dim=-1)
        score_value = (
            logits[GEMMA_YES_ID] - logits[GEMMA_NO_ID]
        ).item()
        answer_mass = (
            probabilities[GEMMA_YES_ID] + probabilities[GEMMA_NO_ID]
        ).item()
        top_values, top_ids = probabilities.topk(5)

        generated = gemma_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            use_cache=True,
        )

    answer = gemma_tokenizer.decode(
        generated[0, input_length:].tolist(),
        skip_special_tokens=False,
    )
    top_tokens = [
        {
            "token": gemma_tokenizer.decode([token_id]),
            "probability": probability,
        }
        for token_id, probability in zip(
            top_ids.tolist(), top_values.tolist()
        )
    ]

    print(f"\nWord: {u.word}")
    print("Generated answer:", repr(answer))
    print(f"Yes-minus-No score: {score_value:.4f}")
    print(f"Probability mass on Yes/No: {answer_mass:.4f}")
    print("Top next tokens:", top_tokens)

    gemma_smoke_results.append({
        "key": list(u.key),
        "messages": messages,
        "input_tokens": input_length,
        "generated_answer": answer,
        "yes_minus_no_score": score_value,
        "yes_no_probability_mass": answer_mass,
        "top_next_tokens": top_tokens,
    })

    del inputs, output, logits, probabilities, generated

save_json(GEMMA_RUN / "smoke_test.json", {
    "model_revision": GEMMA_REVISION,
    "thinking": False,
    "yes_token_id": GEMMA_YES_ID,
    "no_token_id": GEMMA_NO_ID,
    "examples": gemma_smoke_results,
})
print("\nSaved:", GEMMA_RUN / "smoke_test.json")

In [ ]:
import json
import time
import math
import hashlib
import torch
from project_utils import save_json, save_scores, record_environment

GEMMA_BATCH_SIZE = 2
gemma_tokenizer.padding_side = "left"

# Freeze inputs and settings for safe checkpoint reuse.
all_messages = [gemma_messages(u) for u in gemma_dev.usages]
settings = {
    "model_id": GEMMA_ID,
    "revision": GEMMA_REVISION,
    "dtype": "bfloat16",
    "thinking": False,
    "score": "Yes minus No logit",
    "yes_id": GEMMA_YES_ID,
    "no_id": GEMMA_NO_ID,
    "batch_size": GEMMA_BATCH_SIZE,
    "padding_side": "left",
    "truncation": False,
}
fingerprint = hashlib.sha256(
    json.dumps(
        {"settings": settings, "messages": all_messages},
        ensure_ascii=False, sort_keys=True,
    ).encode("utf-8")
).hexdigest()

checkpoint_path = GEMMA_RUN / "inference_checkpoint.json"
records = {}
previous_seconds = 0.0

if checkpoint_path.exists():
    checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
    assert checkpoint["fingerprint"] == fingerprint, (
        "Checkpoint settings differ. Stop rather than mix runs."
    )
    for row in checkpoint["records"]:
        key = tuple(row["key"])
        assert key not in records, f"Duplicate checkpoint key: {key}"
        assert math.isfinite(row["score"])
        assert math.isfinite(row["yes_no_mass"])
        records[key] = row
    previous_seconds = checkpoint["inference_seconds"]

expected_keys = {u.key for u in gemma_dev.usages}
assert set(records) <= expected_keys

pending = [
    (u, messages)
    for u, messages in zip(gemma_dev.usages, all_messages)
    if u.key not in records
]

save_json(GEMMA_RUN / "inference_settings.json", {
    **settings, "fingerprint": fingerprint,
    "n_usages": len(gemma_dev.usages),
})

print(f"Already saved: {len(records)}")
print(f"Remaining: {len(pending)}")

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()

def save_progress():
    save_json(checkpoint_path, {
        "fingerprint": fingerprint,
        "inference_seconds": previous_seconds + time.perf_counter() - started,
        "records": list(records.values()),
    })

try:
    with torch.inference_mode():
        for start in range(0, len(pending), GEMMA_BATCH_SIZE):
            batch_items = pending[start:start + GEMMA_BATCH_SIZE]

            inputs = gemma_processor.apply_chat_template(
                [messages for _, messages in batch_items],
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=False,
                padding=True,
                truncation=False,
            ).to("cuda")

            # Last-position scoring requires left padding.
            assert inputs["attention_mask"][:, -1].all().item()

            output = gemma_model(
                **inputs, use_cache=False, logits_to_keep=1
            )
            logits = output.logits[:, -1, :].float()
            assert torch.isfinite(logits).all().item()

            values = (
                logits[:, GEMMA_YES_ID] - logits[:, GEMMA_NO_ID]
            ).cpu().tolist()
            probabilities = torch.softmax(logits, dim=-1)
            masses = (
                probabilities[:, GEMMA_YES_ID]
                + probabilities[:, GEMMA_NO_ID]
            ).cpu().tolist()
            lengths = inputs["attention_mask"].sum(dim=1).cpu().tolist()

            for (u, _), value, mass, length in zip(
                batch_items, values, masses, lengths
            ):
                records[u.key] = {
                    "key": list(u.key),
                    "score": value,
                    "yes_no_mass": mass,
                    "input_tokens": int(length),
                }

            del inputs, output, logits, probabilities

            if (start // GEMMA_BATCH_SIZE + 1) % 25 == 0:
                save_progress()
                print(
                    f"{len(records)}/{len(gemma_dev.usages)} saved — "
                    f"{time.perf_counter() - started:.1f}s this session",
                    flush=True,
                )
finally:
    # Preserve completed batches even if inference raises an error.
    save_progress()

assert set(records) == expected_keys, "Incomplete inference."
gemma_scores = {key: row["score"] for key, row in records.items()}
save_scores(GEMMA_RUN / "gemma_raw_scores.json", gemma_scores)

record_environment(
    PROJECT, GEMMA_RUN / "inference_manifest.json",
    **settings,
    fingerprint=fingerprint,
    gpu=torch.cuda.get_device_name(0),
    peak_allocated_gpu_gib=torch.cuda.max_memory_allocated() / 2**30,
)

low_mass = sum(row["yes_no_mass"] < 0.90 for row in records.values())
print("\nINFERENCE COMPLETE")
print("Usages scored:", len(gemma_scores))
print("Score range:", min(gemma_scores.values()), max(gemma_scores.values()))
print("Examples with Yes/No probability mass below 0.90:", low_mass)
print("Saved:", GEMMA_RUN / "gemma_raw_scores.json")


In [ ]:
import sys
import subprocess
import numpy as np

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "scikit-learn", "scipy", "bcubed==1.5",
    ],
    check=True,
)

sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "notebooks"))

from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap
from project_utils import (
    read_scores, save_scores, standardized_by_word, frozen,
    new_run, save_json, save_evaluation, record_environment, digest,
)

eval_dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
raw_path = GEMMA_RUN / "gemma_raw_scores.json"
gemma_raw = read_scores(raw_path, eval_dev)
gemma_z = standardized_by_word(eval_dev, gemma_raw)

# Same grids used in the Qwen calibration experiment.
raw_grid = tuple(
    [-1000.0]
    + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)]
    + [1000.0]
)
z_grid = tuple(
    [-1000.0]
    + [round(float(x), 2) for x in np.arange(-12, 12.001, 0.10)]
    + [1000.0]
)

assert all(-40 <= v <= 40 for v in gemma_raw.values())
assert all(-12 <= v <= 12 for v in gemma_z.values())

qwen_zip = (
    PROJECT / "outputs/runs/step04_qwen_calibration"
    / "20261103T171620Z_ab1a34dc/qwen_z.zip"
)
qwen_reference = score(
    eval_dev.require_gold(),
    read_submission(qwen_zip, eval_dev.keys),
)

GEMMA_EVAL = new_run(PROJECT, "step04_gemma4_evaluation")
save_scores(GEMMA_EVAL / "gemma_z_scores.json", gemma_z)
save_json(GEMMA_EVAL / "configuration.json", {
    "source_run": str(GEMMA_RUN),
    "source_scores_sha256": digest(raw_path),
    "qwen_reference": str(qwen_zip),
    "qwen_prediction_sha256": digest(qwen_zip),
    "raw_grid": list(raw_grid),
    "z_grid": list(z_grid),
    "normalization": (
        "Per-word mean/std from unlabeled scores, including held-out inputs"
    ),
    "evaluation": "Separate nested word-CV evaluation for each variant",
})

gemma_evaluations = {}

print(f"Standardized Qwen reference: {qwen_reference['average']:.6f}")

for name, cached_scores, grid in [
    ("gemma_raw", gemma_raw, raw_grid),
    ("gemma_z", gemma_z, z_grid),
]:
    print(f"\n--- {name} ---")

    evaluated = nested_word_cv(
        eval_dev,
        [Candidate(name, frozen(cached_scores), grid)],
    )

    save_evaluation(
        GEMMA_EVAL, name, evaluated, eval_dev, PROJECT,
        official=True,
    )
    print("Official scorer agreement: PASS")

    difference = word_bootstrap(
        evaluated["report"], other=qwen_reference
    )
    save_json(GEMMA_EVAL / f"{name}_vs_qwen_z.json", difference)
    print(
        f"Difference vs standardized Qwen: "
        f"{difference['estimate']:+.4f} "
        f"[{difference['lower']:+.4f}, {difference['upper']:+.4f}]"
    )

    print("Held-out thresholds:")
    for selection in evaluated["selections"]:
        print(
            selection["test_group"],
            "→", float(selection["threshold"]),
        )

    gemma_evaluations[name] = evaluated

record_environment(
    PROJECT, GEMMA_EVAL / "manifest.json",
    purpose="Gemma raw and standardized held-out evaluation",
    model_revision=GEMMA_REVISION,
    source_run=str(GEMMA_RUN),
    new_model_inference=False,
)

print("\nSaved:", GEMMA_EVAL)

In [ ]:
import json
import random
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from semeval_task3.evaluation import read_submission
from project_utils import save_json

gold = eval_dev.require_gold()
gemma_predictions = read_submission(
    GEMMA_EVAL / "gemma_raw.zip", eval_dev.keys
)
qwen_predictions = read_submission(qwen_zip, eval_dev.keys)

saved_evaluation = json.loads(
    (GEMMA_EVAL / "gemma_raw.json").read_text(encoding="utf-8")
)
threshold_by_group = {
    tuple(s["test_group"]): float(s["threshold"])
    for s in saved_evaluation["selections"]
}

word_results = []
all_errors = []
review_sample = []
rng = random.Random(2027)

print(
    f"{'Word':<12} {'N':>4} {'Prec':>6} {'Recall':>6} "
    f"{'Gemma':>6} {'Qwen':>6} {'All+':>6} "
    f"{'ΔQwen':>7} {'AUC':>6} {'FP':>4} {'FN':>4}"
)

for group in sorted({u.group for u in eval_dev.usages}):
    usages = [u for u in eval_dev.usages if u.group == group]
    y = np.array([gold[u.key] for u in usages])
    pred = np.array([gemma_predictions[u.key] for u in usages])
    qpred = np.array([qwen_predictions[u.key] for u in usages])
    scores = np.array([gemma_raw[u.key] for u in usages])

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", zero_division=0
    )
    _, _, qwen_f1, _ = precision_recall_fscore_support(
        y, qpred, average="binary", zero_division=0
    )

    prevalence = float(y.mean())
    all_positive_f1 = 2 * prevalence / (1 + prevalence)
    auc = float(roc_auc_score(y, scores)) if len(set(y)) == 2 else None
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())

    word_results.append({
        "language": group[0],
        "word": group[1],
        "n": len(usages),
        "gold_positive_rate": prevalence,
        "predicted_positive_rate": float(pred.mean()),
        "precision": float(precision),
        "recall": float(recall),
        "gemma_f1": float(f1),
        "qwen_f1": float(qwen_f1),
        "all_positive_f1": all_positive_f1,
        "difference_vs_qwen": float(f1 - qwen_f1),
        "difference_vs_all_positive": float(f1 - all_positive_f1),
        "auc": auc,
        "false_positives": fp,
        "false_negatives": fn,
        "held_out_threshold": threshold_by_group[group],
    })

    auc_text = f"{auc:.3f}" if auc is not None else "N/A"
    print(
        f"{group[1]:<12} {len(usages):>4} "
        f"{precision:>6.3f} {recall:>6.3f} "
        f"{f1:>6.3f} {qwen_f1:>6.3f} {all_positive_f1:>6.3f} "
        f"{f1-qwen_f1:>+7.3f} {auc_text:>6} {fp:>4} {fn:>4}"
    )

    word_errors = []
    for u in usages:
        prediction = gemma_predictions[u.key]
        if prediction == gold[u.key]:
            continue

        word_errors.append({
            "key": list(u.key),
            "word": u.word,
            "year": str(u.year),
            "marked_text": u.marked_text(),
            "definition": u.definition,
            "gold": gold[u.key],
            "prediction": prediction,
            "qwen_prediction": qwen_predictions[u.key],
            "score": float(gemma_raw[u.key]),
            "threshold": threshold_by_group[group],
            "error_type": "FP" if prediction == 1 else "FN",
            "manual_category": "",
            "review_notes": "",
        })

    all_errors.extend(word_errors)

    # Up to three examples of each error type per word.
    for error_type in ["FP", "FN"]:
        candidates = [
            e for e in word_errors if e["error_type"] == error_type
        ]
        review_sample.extend(
            rng.sample(candidates, min(3, len(candidates)))
        )

save_json(GEMMA_EVAL / "per_word_diagnostics.json", word_results)
save_json(GEMMA_EVAL / "all_errors.json", all_errors)
save_json(GEMMA_EVAL / "error_review_sample.json", {
    "sampling": "Seed 2027; up to 3 FP and 3 FN per word",
    "note": "Balanced review sample, not representative error frequencies",
    "examples": review_sample,
})

wins_qwen = sum(r["difference_vs_qwen"] > 0 for r in word_results)
wins_all = sum(r["difference_vs_all_positive"] > 0 for r in word_results)

print(f"\nWords improved over Qwen: {wins_qwen}/{len(word_results)}")
print(f"Words improved over all-positive: {wins_all}/{len(word_results)}")
print("Total false positives:", sum(r["false_positives"] for r in word_results))
print("Total false negatives:", sum(r["false_negatives"] for r in word_results))
print("Errors saved:", len(all_errors))
print("Review examples saved:", len(review_sample))
print("Saved in:", GEMMA_EVAL)

In [ ]:
import json
from pathlib import Path
# GEMMA_RUN is defined in cell 1; if the runtime restarted, set it by hand:
# GEMMA_RUN = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67"

raw = json.loads((GEMMA_RUN / "gemma_raw_scores.json").read_text())
smoke = json.loads((GEMMA_RUN / "smoke_test.json").read_text())

print("1) Batched vs single-example scores (should match within ~0.5):")
for ex in smoke["examples"]:
    k = "|".join(ex["key"])
    print(f"   {ex['key'][1]:<10} single={ex['yes_minus_no_score']:.3f}  batched={raw[k]:.3f}  "
          f"diff={raw[k] - ex['yes_minus_no_score']:+.3f}")

ck = json.loads((GEMMA_RUN / "inference_checkpoint.json").read_text())
sec = ck["inference_seconds"]
print(f"\n2) Inference time: {sec/60:.1f} min total, {1000*sec/1983:.0f}s per 1,000 usages (A100, batch 2)")

In [ ]:
import json
from project_utils import save_json

review_path = GEMMA_EVAL / "error_review_sample.json"
review = json.loads(review_path.read_text(encoding="utf-8"))
examples = review["examples"]

priority_words = ["dumpa", "kvinna", "skär", "fru", "herre"]
priority_examples = []

for word in priority_words:
    for error_type in ["FP", "FN"]:
        matches = [
            example for example in examples
            if example["word"] == word
            and example["error_type"] == error_type
        ]
        if matches:
            priority_examples.append(matches[0])

for number, example in enumerate(priority_examples, start=1):
    meaning = (
        "Gemma says YES; reference label is NO"
        if example["error_type"] == "FP"
        else "Gemma says NO; reference label is YES"
    )

    print("=" * 80)
    print(
        f"Example {number} | {example['word']} | "
        f"Year: {example['year']} | {example['error_type']}"
    )
    print("ID:", " | ".join(map(str, example["key"])))
    print(meaning)
    print(
        f"Score: {example['score']:.4f} | "
        f"Held-out threshold: {example['threshold']:.4f}"
    )
    print("\nTARGET DEFINITION:")
    print(example["definition"])
    print("\nSENTENCE — target marked with <t>...</t>:")
    print(example["marked_text"])
    print()

save_json(GEMMA_EVAL / "priority_error_review.json", {
    "source": str(review_path),
    "selection": (
        "First saved example per FP/FN category for each priority word; "
        "qualitative inspection, not representative error frequencies"
    ),
    "examples": priority_examples,
})

print("=" * 80)
print("Examples displayed:", len(priority_examples))
print("Full saved review sample:", len(examples))
print("No predictions, thresholds, or labels were changed.")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

# Use the completed evaluation folder.
inspection_dir = (
    PROJECT / "outputs/runs/step04_gemma4_evaluation"
    / "20261105T170910Z_b3fc303a"
)

report_files = {
    "gemma_raw_evaluation": "gemma_raw.json",
    "gemma_standardized_evaluation": "gemma_z.json",
    "raw_vs_qwen": "gemma_raw_vs_qwen_z.json",
    "standardized_vs_qwen": "gemma_z_vs_qwen_z.json",
    "per_word_diagnostics": "per_word_diagnostics.json",
    "all_errors": "all_errors.json",
    "review_sample": "error_review_sample.json",
    "evaluation_configuration": "configuration.json",
    "evaluation_manifest": "manifest.json",
}

bundle = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Consolidated inspection of the completed Gemma experiment",
    "notes": [
        "Predictions are thresholded decisions, not generated answers.",
        "A positive Yes-minus-No score can still fall below the fitted threshold.",
        "The review sample is balanced for inspection, not representative.",
        "These are Swedish development results, not official test results.",
    ],
}

for section, filename in report_files.items():
    path = inspection_dir / filename
    assert path.is_file(), f"Missing file: {path}"
    bundle[section] = json.loads(path.read_text(encoding="utf-8"))

# Include scores for every usage, including correctly classified ones.
source_run = Path(bundle["evaluation_configuration"]["source_run"])
raw_scores_path = source_run / "gemma_raw_scores.json"
assert raw_scores_path.is_file(), f"Missing: {raw_scores_path}"
bundle["all_raw_scores"] = json.loads(
    raw_scores_path.read_text(encoding="utf-8")
)

output_path = inspection_dir / "gemma_complete_inspection.json"
output_path.write_text(
    json.dumps(bundle, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Errors included:", len(bundle["all_errors"]))
print("Review examples:", len(bundle["review_sample"]["examples"]))
print("Word reports:", len(bundle["per_word_diagnostics"]))
print("Raw scores:", len(bundle["all_raw_scores"]))
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print("Saved:", output_path)

files.download(str(output_path))

In [ ]:
import json, numpy as np
from semeval_task3.evaluation import score
from project_utils import save_json, read_scores, digest

# GEMMA_RUN / GEMMA_EVAL / GEMMA_ID / GEMMA_REVISION come from earlier cells.
# If the runtime restarted, set them by hand:
# GEMMA_RUN  = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67"
# GEMMA_EVAL = PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a"
# GEMMA_ID, GEMMA_REVISION = "google/gemma-4-31B-it", "842da3794eaa0b77d5f08bae87a17459d91ff475"

gold = eval_dev.require_gold()
raw = read_scores(GEMMA_RUN / "gemma_raw_scores.json", eval_dev)
grid = tuple([-1000.0] + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)] + [1000.0])

# Final threshold for test: best mean per-word F1 on all 10 words; ties -> lower threshold (harness rule)
best_f1, best_t = -1.0, None
for t in sorted(set(grid)):
    f = score(gold, {k: int(v >= t) for k, v in raw.items()})["average"]
    if f > best_f1 + 1e-12:
        best_f1, best_t = f, t

nested = json.loads((GEMMA_EVAL / "gemma_raw.json").read_text())
nested_f1 = nested["report"]["average"]
ci = nested["confidence_interval"]
vs_ap = nested["versus_all_positive"]
ck = json.loads((GEMMA_RUN / "inference_checkpoint.json").read_text())

frozen_config = {
    "status": "provisional primary system (Step 4 not complete; no multilingual validation yet)",
    "model_id": GEMMA_ID,
    "model_revision": GEMMA_REVISION,
    "dtype": "bfloat16",
    "thinking": False,
    "system_prompt": "You are an expert lexicographer of historical Swedish.",
    "user_prompt_template": (
        "Target word (lemma): {word}\nSense definition (Swedish): {definition}\n"
        "Sentence from {year}: {marked_text}\n\nDoes the word marked with <t> and </t> in the sentence "
        "express the sense given by the definition? Answer only Yes or No."),
    "score": "logit(Yes) - logit(No) at the first generated position",
    "decision_rule": "predict 1 if score >= threshold",
    "final_threshold": best_t,
    "final_threshold_source": "all 10 Swedish dev words, same grid and tie-break as nested CV",
    "in_sample_f1_at_final_threshold": best_f1,
    "honest_estimate_nested_cv_f1": nested_f1,
    "nested_cv_ci": [ci["lower"], ci["upper"]],
    "nested_cv_vs_all_positive": [vs_ap["estimate"], vs_ap["lower"], vs_ap["upper"]],
    "seconds_per_1000_usages_a100_batch2": 1000 * ck["inference_seconds"] / len(raw),
    "source_scores": str(GEMMA_RUN / "gemma_raw_scores.json"),
    "source_scores_sha256": digest(GEMMA_RUN / "gemma_raw_scores.json"),
    "nested_evaluation": str(GEMMA_EVAL),
}
FROZEN = PROJECT / "outputs/frozen/provisional_primary_gemma4_31b.json"
save_json(FROZEN, frozen_config)

doc = f"""# Step 4: Gemma 4 31B result (provisional primary)

| System | Mean per-word F1 (nested CV) | vs all-positive |
|---|---:|---|
| Gemma 4 31B, raw | {nested_f1:.4f} | {vs_ap['estimate']:+.4f} [{vs_ap['lower']:+.4f}, {vs_ap['upper']:+.4f}] |
| Qwen3-8B, z-scored | 0.4761 | +0.0045 [-0.0386, +0.0397] |
| All-positive | 0.4716 | |

- Same prompt as Qwen, fixed before Gemma was run. Thinking off. Revision {GEMMA_REVISION}.
- Nested-CV thresholds 10.0 to 11.25 (10.5 in 7 of 10 folds). Final threshold for test: {best_t}.
- In-sample F1 at the final threshold ({best_f1:.4f}) is optimistic; the honest estimate is the nested-CV value.
- Batched vs single-example scores differ by at most 0.2 (bf16 rounding).
- Weak spot: fru drops below all-positive (0.852 vs 0.944) under one global threshold.
- Limits: Swedish only, 10 words; prompt names Swedish and must be templated for other languages.

Frozen configuration: {FROZEN}
"""
(PROJECT / "docs/STEP04_GEMMA_RESULTS.md").write_text(doc, encoding="utf-8")

print(f"Final threshold for test: {best_t}  (in-sample F1 {best_f1:.4f}; honest nested estimate {nested_f1:.4f})")
print("Frozen config:", FROZEN)
print("Doc:", PROJECT / "docs/STEP04_GEMMA_RESULTS.md")

In [ ]:
import json, re, torch
from project_utils import new_run, save_json

V2_RUN = new_run(PROJECT, "step06_gemma_contrastive_v2")
LANG = {"SV": "Swedish", "EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "RU": "Russian"}

def sense_list_messages(word, definition, lang):
    return [{"role": "user", "content": (
        f"Word: {word} ({LANG[lang]})\n"
        f"Target sense definition: {definition}\n\n"
        f"List other senses of this {LANG[lang]} word, attested in texts from 1880 to today, that are clearly "
        "different from the target sense, including closely related, extended or figurative senses. "
        "Do not restate, narrow or split the target sense. "
        f"Write each as a short dictionary definition in {LANG[lang]}. "
        "Return a JSON list of 2 to 5 strings and nothing else.")}]

other_senses = {}
for lang, word, definition in sorted({(u.language, u.word, u.definition) for u in gemma_dev.usages}):
    inputs = gemma_processor.apply_chat_template(
        sense_list_messages(word, definition, lang), tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True, enable_thinking=False).to("cuda")
    with torch.inference_mode():
        out = gemma_model.generate(**inputs, max_new_tokens=400, do_sample=False)
    text = gemma_tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    m = re.search(r"\[.*\]", text, re.S)
    senses = json.loads(m.group(0)) if m else None
    assert isinstance(senses, list) and 1 <= len(senses) <= 6 and all(isinstance(s, str) and s.strip() for s in senses), \
        f"Parse failed for {word}: {text!r}"
    other_senses[(lang, word)] = [s.strip() for s in senses]
    print(f"\n{word}\n  TARGET: {definition}")
    for s in other_senses[(lang, word)]:
        print("  other:", s)

save_json(V2_RUN / "other_senses.json", {f"{l}|{w}": s for (l, w), s in other_senses.items()})
print("\nSaved:", V2_RUN / "other_senses.json")

In [ ]:
import json
import time
import hashlib
import torch
from tqdm.auto import tqdm
from project_utils import new_run, save_json

# Check that Notebook 4b still has the loaded model and data.
for name in ["PROJECT", "gemma_model", "gemma_processor",
             "gemma_tokenizer", "gemma_dev"]:
    assert name in globals(), f"Missing {name}. Run the existing setup first."

assert torch.cuda.is_available(), "A GPU runtime is required."
assert all(u.language == "SV" for u in gemma_dev.usages)

DIRECT_SYSTEM = "You are an expert lexicographer of historical Swedish."

DIRECT_INSTRUCTIONS = """Determine whether the marked word expresses the supplied
dictionary sense in this particular sentence.

Evaluate the meaning of the marked word, not merely the topic of the sentence
or facts about the person or object it refers to.
Respect the distinctions and qualifications in the definition.
A related meaning is not sufficient by itself.
Consider grammatical use and the surrounding historical context.
Figurative usage is not automatically a match or a mismatch: judge whether
the supplied sense still applies.
The supplied sense need not be the only applicable interpretation.

Answer only Yes or No."""

def direct_messages(u):
    return [
        {"role": "system", "content": DIRECT_SYSTEM},
        {"role": "user", "content": (
            f"Target word (lemma): {u.word}\n"
            f"Sense definition (Swedish): {u.definition}\n"
            f"Sentence from {u.year}: {u.marked_text()}\n\n"
            f"{DIRECT_INSTRUCTIONS}"
        )}
    ]

yes_ids = gemma_tokenizer.encode("Yes", add_special_tokens=False)
no_ids = gemma_tokenizer.encode("No", add_special_tokens=False)
assert len(yes_ids) == len(no_ids) == 1
yes_id, no_id = yes_ids[0], no_ids[0]

usages = list(gemma_dev.usages)
keys = ["|".join(u.key) for u in usages]
assert len(keys) == len(set(keys)), "Duplicate usage IDs."

# Fingerprint the actual prompts so a changed prompt cannot reuse old scores.
prompt_hash = hashlib.sha256(
    json.dumps(
        [(k, direct_messages(u)) for k, u in zip(keys, usages)],
        ensure_ascii=False, sort_keys=True
    ).encode()
).hexdigest()

# Re-running this cell in the same session resumes this exact experiment.
if (
    "DIRECT_RUN" not in globals()
    or globals().get("DIRECT_PROMPT_HASH") != prompt_hash
):
    DIRECT_RUN = new_run(PROJECT, "step04_gemma_direct_v2")
    DIRECT_PROMPT_HASH = prompt_hash

checkpoint = DIRECT_RUN / "checkpoint.json"
records = json.loads(checkpoint.read_text()) if checkpoint.exists() else {}
assert set(records).issubset(set(keys)), "Unexpected IDs in checkpoint."

save_json(DIRECT_RUN / "configuration.json", {
    "system_prompt": DIRECT_SYSTEM,
    "instructions": DIRECT_INSTRUCTIONS,
    "prompt_sha256": prompt_hash,
    "model": getattr(gemma_model.config, "_name_or_path", None),
    "model_revision": getattr(gemma_model.config, "_commit_hash", None),
    "thinking": False,
    "score": "Yes logit minus No logit",
    "yes_token_id": yes_id,
    "no_token_id": no_id,
    "batch_size": 2,
    "examples": len(usages),
    "uses_generated_alternative_senses": False,
    "note": "Exploratory prompt revision informed by development error review."
})

gemma_processor.tokenizer.padding_side = "left"
gemma_tokenizer.padding_side = "left"
gemma_model.eval()

pending = [(k, u) for k, u in zip(keys, usages) if k not in records]
print(f"Run: {DIRECT_RUN}")
print(f"Already saved: {len(records)}; remaining: {len(pending)}")

started = time.perf_counter()

try:
    for start in tqdm(range(0, len(pending), 2), desc="Scoring revised prompt"):
        batch = pending[start:start + 2]

        inputs = gemma_processor.apply_chat_template(
            [direct_messages(u) for _, u in batch],
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            enable_thinking=False,
            processor_kwargs={"padding": True, "truncation": False},
        ).to("cuda")

        with torch.inference_mode():
            output = gemma_model(
                **inputs, use_cache=False, logits_to_keep=1
            )
            logits = output.logits[:, -1, :].float()
            margins = logits[:, yes_id] - logits[:, no_id]
            probabilities = torch.softmax(logits, dim=-1)
            answer_mass = probabilities[:, yes_id] + probabilities[:, no_id]

        assert torch.isfinite(margins).all(), "Non-finite model score."

        for i, (key, _) in enumerate(batch):
            records[key] = {
                "score": float(margins[i].item()),
                "yes_no_mass": float(answer_mass[i].item()),
                "input_tokens": int(inputs["attention_mask"][i].sum().item()),
            }

        del inputs, output, logits, margins, probabilities, answer_mass

        if (start // 2 + 1) % 25 == 0:
            save_json(checkpoint, records)
finally:
    save_json(checkpoint, records)

assert set(records) == set(keys), "Incomplete scoring; rerun to resume."

DIRECT_SCORES = {key: records[key]["score"] for key in keys}
save_json(DIRECT_RUN / "gemma_raw_scores.json", DIRECT_SCORES)

low_mass = sum(r["yes_no_mass"] < 0.90 for r in records.values())
save_json(DIRECT_RUN / "inference_summary.json", {
    "examples": len(records),
    "seconds_this_execution": time.perf_counter() - started,
    "minimum_score": min(DIRECT_SCORES.values()),
    "maximum_score": max(DIRECT_SCORES.values()),
    "examples_with_yes_no_mass_below_0_90": low_mass,
})

print(f"\nCOMPLETE: {len(records)} scores")
print(f"Time this execution: {time.perf_counter() - started:.1f}s")
print(f"Examples with Yes/No mass below 0.90: {low_mass}")
print("Saved:", DIRECT_RUN / "gemma_raw_scores.json")
print("Next: held-out-word evaluation. No F1 has been calculated yet.")

In [ ]:
import json
import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")

for folder in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from project_utils import (
    new_run, read_scores, frozen, save_json,
    save_evaluation, digest, record_environment
)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import (
    read_submission, score, word_bootstrap
)

# 1. Load the labeled dataset specifically for evaluation.
eval_dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = eval_dev.require_gold()
print(f"Loaded {len(gold)} gold labels for evaluation.")

source_run = PROJECT / (
    "outputs/runs/step04_gemma_direct_v2/"
    "20261105T173434Z_34f0b934"
)
original_run = PROJECT / (
    "outputs/runs/step04_gemma4_evaluation/"
    "20261105T170910Z_b3fc303a"
)

# 2. Load revised-prompt scores and validate exact dataset coverage.
scores_path = source_run / "gemma_raw_scores.json"
new_scores = read_scores(scores_path, eval_dev)

print(
    f"Revised score range: "
    f"{min(new_scores.values()):.4f} to {max(new_scores.values()):.4f}"
)

# 3. Reproduce the original result using the same labels.
original_zip = original_run / "gemma_raw.zip"
original_predictions = read_submission(
    original_zip, set(eval_dev.keys)
)
original_report = score(gold, original_predictions)

assert abs(original_report["average"] - 0.6782392862783919) < 1e-10, (
    "Original F1 does not match the saved reference. "
    "Stop and inspect the dataset before comparing."
)
print(f"Original Gemma F1: {original_report['average']:.6f}")

# 4. Reuse the original threshold grid.
original_config = json.loads(
    (original_run / "configuration.json").read_text(encoding="utf-8")
)
thresholds = tuple(float(t) for t in original_config["raw_grid"])

assert min(thresholds) < min(new_scores.values())
assert max(thresholds) > max(new_scores.values())

DIRECT_EVAL = new_run(
    PROJECT, "step04_gemma_direct_v2_evaluation"
)

save_json(DIRECT_EVAL / "configuration.json", {
    "source_run": str(source_run),
    "source_scores_sha256": digest(scores_path),
    "original_reference": str(original_zip),
    "original_predictions_sha256": digest(original_zip),
    "threshold_grid": list(thresholds),
    "evaluation": "Nested leave-one-word-out; raw scores only",
    "limitation": (
        "The prompt was revised after inspecting development errors. "
        "This comparison is exploratory. Bootstrap intervals do not "
        "account for that adaptation or repeated experiments."
    )
})

# 5. Select thresholds without using the held-out word's labels.
print("\nRunning nested leave-one-word-out evaluation...")

direct_result = nested_word_cv(
    eval_dev,
    [Candidate(
        "gemma_direct_v2_raw",
        frozen(new_scores),
        thresholds
    )]
)

# Save predictions, metrics, confidence intervals, and official scorer check.
save_evaluation(
    DIRECT_EVAL,
    "gemma_direct_v2_raw",
    direct_result,
    eval_dev,
    PROJECT,
    official=True
)
print("Official scorer agreement: PASS")

# 6. Compare both systems using a paired bootstrap over words.
difference = word_bootstrap(
    direct_result["report"],
    other=original_report
)
save_json(
    DIRECT_EVAL / "difference_vs_original_gemma.json",
    difference
)

print(
    "\nRevised minus original Gemma: "
    f"{100 * difference['estimate']:+.2f} F1 points\n"
    "95% paired interval: "
    f"[{100 * difference['lower']:+.2f}, "
    f"{100 * difference['upper']:+.2f}]"
)

# 7. Show which words improved or worsened.
old_by_word = {
    (row["language"], row["word"]): row
    for row in original_report["per_word"]
}

print(
    f"\n{'Word':12} {'Original':>9} {'Revised':>9} "
    f"{'Change':>9} {'FP':>5} {'FN':>5}"
)

for row in direct_result["report"]["per_word"]:
    old = old_by_word[(row["language"], row["word"])]
    change = row["f1"] - old["f1"]
    print(
        f"{row['word']:12} "
        f"{old['f1']:9.4f} "
        f"{row['f1']:9.4f} "
        f"{change:+9.4f} "
        f"{row['fp']:5d} "
        f"{row['fn']:5d}"
    )

print("\nHeld-out thresholds:")
for selection in direct_result["selections"]:
    print(
        tuple(selection["test_group"]),
        "→",
        selection["threshold"]
    )

record_environment(
    PROJECT,
    DIRECT_EVAL / "manifest.json",
    purpose="Revised versus original Gemma prompt evaluation",
    new_model_inference=False
)

print("\nEVALUATION COMPLETE")
print("Saved:", DIRECT_EVAL)

In [ ]:
import json, time, numpy as np, torch
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from project_utils import new_run, save_json, record_environment, read_scores
from semeval_task3.data import load_dataset

# --- Frozen Swedish threshold (written by the freeze cell) ---
frozen_path = PROJECT / "outputs/frozen/provisional_primary_gemma4_31b.json"
assert frozen_path.is_file(), "Frozen config missing: run the freeze cell first."
SV_T = float(json.loads(frozen_path.read_text())["final_threshold"])
print("Frozen Swedish threshold:", SV_T)

# --- English WiC-TSV, prepared in Step 3 ---
wic_dir = PROJECT / "outputs/runs/step03_english_data/20261103T165602Z_11f07da3"
wic = {s: json.loads((wic_dir / f"prepared_{s}.json").read_text()) for s in ("train", "dev")}
print({s: len(r) for s, r in wic.items()})

# Same prompt as Gemma v1, language set to English. WiC-TSV has no year, so that line reads "Sentence:".
def en_messages(r):
    return [{"role": "system", "content": "You are an expert lexicographer of historical English."},
            {"role": "user", "content": (
                f"Target word (lemma): {r['word']}\n"
                f"Sense definition (English): {r['definition']}\n"
                f"Sentence: {r['marked_text']}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]

# --- Score all English pairs ---
gemma_tokenizer.padding_side = "left"
EN_RUN = new_run(PROJECT, "step05_threshold_transfer_english")
scores, t0 = {}, time.time()
with torch.inference_mode():
    for split, rows in wic.items():
        for i in range(0, len(rows), 4):
            batch = rows[i:i + 4]
            inputs = gemma_processor.apply_chat_template(
                [en_messages(r) for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True, enable_thinking=False,
                processor_kwargs={"padding": True, "truncation": False}).to("cuda")
            assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
            logits = gemma_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
            for r, v in zip(batch, (logits[:, GEMMA_YES_ID] - logits[:, GEMMA_NO_ID]).tolist()):
                scores[r["id"]] = v
        print(f"{split}: {len(rows)} scored, {time.time() - t0:.0f}s")
save_json(EN_RUN / "gemma_v1_english_scores.json", scores)

# --- Compare cutoffs ---
def arrays(split):
    return (np.array([r["label"] for r in wic[split]]), np.array([scores[r["id"]] for r in wic[split]]))
def prf(y, s, t):
    p, r, f, _ = precision_recall_fscore_support(y, (s >= t).astype(int), average="binary", zero_division=0)
    return float(p), float(r), float(f)

grid = [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)]
ytr, s_tr = arrays("train"); ydv, s_dv = arrays("dev")
en_t = max(grid, key=lambda t: (prf(ytr, s_tr, t)[2], -t))       # tuned on English train; ties -> lower
oracle_t = max(grid, key=lambda t: (prf(ydv, s_dv, t)[2], -t))   # ceiling, reference only

# --- Score scale: Swedish vs English, separately for gold Yes and gold No (independent of base rate) ---
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
sv_scores = read_scores(PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json", sv)
sv_y = np.array([sv.gold[u.key] for u in sv.usages]); sv_s = np.array([sv_scores[u.key] for u in sv.usages])
en_y = np.concatenate([ytr, ydv]); en_s = np.concatenate([s_tr, s_dv])

res = {
    "swedish_threshold": SV_T, "english_train_threshold": en_t, "dev_oracle_threshold": oracle_t,
    "dev_auc": float(roc_auc_score(ydv, s_dv)),
    "dev_at_swedish_threshold": prf(ydv, s_dv, SV_T),
    "dev_at_english_threshold": prf(ydv, s_dv, en_t),
    "dev_oracle": prf(ydv, s_dv, oracle_t),
    "dev_yes_rate_at_swedish_threshold": float((s_dv >= SV_T).mean()),
    "dev_positive_rate": float(ydv.mean()),
    "median_score_gold_yes": {"swedish": float(np.median(sv_s[sv_y == 1])), "english": float(np.median(en_s[en_y == 1]))},
    "median_score_gold_no": {"swedish": float(np.median(sv_s[sv_y == 0])), "english": float(np.median(en_s[en_y == 0]))},
}
save_json(EN_RUN / "threshold_transfer.json", res)
record_environment(PROJECT, EN_RUN / "manifest.json", purpose="Threshold transfer check on English WiC-TSV",
                   model_revision=GEMMA_REVISION, swedish_threshold=SV_T)

print(f"\nEnglish WiC-TSV dev ({len(ydv)} pairs): positive rate {res['dev_positive_rate']:.2f}, Gemma AUC {res['dev_auc']:.3f}")
for name, key, t in [("Swedish cutoff (what we'd use on test)", "dev_at_swedish_threshold", SV_T),
                     ("English-tuned cutoff", "dev_at_english_threshold", en_t),
                     ("Best possible (ceiling)", "dev_oracle", oracle_t)]:
    p, r, f = res[key]
    print(f"  {name:<40} t={t:<7} F1={f:.3f}  P={p:.3f}  R={r:.3f}")
gap = res["dev_at_english_threshold"][2] - res["dev_at_swedish_threshold"][2]
print(f"\nCost of reusing the Swedish cutoff on English: {gap:+.3f} F1")
print("Median score, gold YES:", res["median_score_gold_yes"])
print("Median score, gold NO: ", res["median_score_gold_no"])
print("References on the same English dev set: graded-wsd 0.810, all-positive 0.675")